# EZStats Pipeline v3

Run setup cells 1-5, then run each match separately. Messi is first. Each match saves its video, JSON files, and full log to Drive before its cell completes. This is a candidate pipeline to validate, not a measured accuracy guarantee.


### 1. GPU


In [ ]:
import torch
print(torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime before running the videos.'


### 2. Drive


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/ezstats')
assert DRIVE.is_dir()


### 3. Code

Local edits must be committed and pushed before this cell can fetch them. Pulling code does not update an already-open notebook; open this v3 notebook separately.


In [ ]:
import subprocess
REPO = Path('/content/EZStatsAIWorker')
if REPO.exists():
    subprocess.run(['git', 'checkout', 'test'], cwd=REPO, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', '-b', 'test', 'https://github.com/MattyKKS/EZStatsAIWorker.git', str(REPO)], check=True)
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO, check=True)
assert (REPO / 'run_pipeline_v3.py').exists(), 'Push the v3 changes first.'


### 4. Dependencies


In [ ]:
import os, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO) + '[ml,appearance]', 'ultralytics==8.4.48', 'supervision==0.25.1', 'lap>=0.5.12', 'git+https://github.com/roboflow/sports.git@42c80c06b6b65a7f89455b89fe31cdf4c38ba227'], check=True)
os.environ['PYTHONPATH'] = str(REPO / 'src')
subprocess.run([sys.executable, '-c', 'import ez_worker, ultralytics, supervision, sports; print(ultralytics.__version__, supervision.__version__)'], cwd=REPO, check=True)


### 5. Models and videos

Models are copied to runtime storage. The run helper also copies each selected video locally before processing it, avoiding repeated reads through Drive.


In [ ]:
import shutil
for name in ('artifacts', 'data'):
    assert (DRIVE / name).is_dir(), str(DRIVE / name)
models = REPO / 'artifacts'
if models.is_symlink():
    models.unlink()
shutil.copytree(DRIVE / 'artifacts', models, dirs_exist_ok=True)
video_link = REPO / 'data'
if not video_link.exists():
    video_link.symlink_to(DRIVE / 'data', target_is_directory=True)
print('Videos:', [p.name for p in (video_link / 'raw').glob('*.mp4')])


### 6A. Messi


In [ ]:
subprocess.run([sys.executable, '-u', str(REPO / 'scripts/run_colab_clip.py'), 'leo_messi_30pass.mp4'], cwd=REPO, check=True)


### 6B. Original benchmark


In [ ]:
subprocess.run([sys.executable, '-u', str(REPO / 'scripts/run_colab_clip.py'), '08fd33_4.mp4'], cwd=REPO, check=True)


### 6C. Brighton


In [ ]:
subprocess.run([sys.executable, '-u', str(REPO / 'scripts/run_colab_clip.py'), 'BrightonGoal.mp4'], cwd=REPO, check=True)


### 6D. Mason Mount


In [ ]:
subprocess.run([sys.executable, '-u', str(REPO / 'scripts/run_colab_clip.py'), '19PassesAndMasonGoal.mp4'], cwd=REPO, check=True)


### Resume an interrupted finalization

The printed run directory contains raw tracks and crops after detection finishes. To redo team assignment, events, and rendering without rerunning detection, use `python run_pipeline_v3.py --resume outputs/<run>`. Results from a Colab runtime still need to be copied to Drive. Retain the original run log and `run_manifest.json` when comparing results.
